# Complete OceanBench Workflow

This notebook is a **complete, guided workflow** to browse datasets, define a request (variables/region/time), fetch & cache data, and visualize it with correct time/depth handling.


## What This Notebook Does

1. **Browse all available datasets** - See what ocean products are available
2. **Explore dataset details** - Understand time coverage, variables, and spatial resolution
3. **Select your variables** - Choose which ocean parameters to fetch
4. **Define your region** - Specify geographic bounds (longitude/latitude)
5. **Choose time window** - Select start and end dates based on dataset availability
6. **Estimate data size** - Check download size before fetching
7. **Fetch and cache data** - Download, sanitize, and store data locally
8. **Visualize with proper time labels** - Plot maps with actual dates (not indices)
9. **Create movies** - Generate animated GIFs/MP4s showing temporal evolution


## How to use this notebook

- Read each markdown step, then run the cell(s) directly below it.
- User-editable parameters are always under a **`# USER INPUT`** header.
- If you choose a preset scenario, you can skip several steps.


## Prerequisites

- **Python 3.9+**
- **Internet access** (for downloads)
- **Disk space** (Step 7 estimates size before you fetch)
- **Credentials** (only for some products, e.g. Copernicus Marine)


## Installation

Before running this notebook, install OceanBench Data Provider:

```bash
pip install oceanbench-data-provider
```

For notebooks and movie generation, install the optional dependencies:

```bash
pip install "oceanbench-data-provider[notebooks]"  # for Jupyter notebooks
pip install "oceanbench-data-provider[movie]"  # for movie generation
pip install "oceanbench-data-provider[notebooks,movie]"  # for both
```


## Step 1: Import Libraries and Initialize

Run the cell below to import OceanBench and helper visualization functions.


In [ ]:
from oceanbench_data_provider import DataProvider, list_products, describe
from oceanbench_data_provider.viz import quicklook_map, make_movie, plot_region_bounds, interactive_globe
from oceanbench_data_provider.scenarios import list_scenarios, get_scenario, list_regions, get_region_bounds, get_region
from oceanbench_data_provider.cache.store import CacheStore
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

# Initialize the data provider
provider = DataProvider()

: 

## Step 2: Browse All Available Datasets

List all available products (datasets) so you can pick one by ID.

- **HYCOM**: Global Reanalysis (1994–2015), Global Analysis (2014–2024)
- **Copernicus Marine (Physical)**: Reanalysis (1993–2025), Analysis (2022–now + 10-day forecast)
- **Copernicus Marine (BGC)**: Reanalysis (1993–2025), Analysis (2021–now + 10-day forecast)


In [ ]:
# List all available products
products = list_products()

print("Available OceanBench Products:")
print("=" * 50)
for i, p in enumerate(products, 1):
    print(f"{i:2d}. {p}")

print(f"\nTotal: {len(products)} products available")

## Step 2a (Optional): Use a Preset Scenario

You can **skip manual product + variable + region + time selection** by choosing a preset scenario. Scenarios define a product, region, time window, and variables in one go. If you use a scenario, you can skip to **Step 7**.



In [ ]:
# List available preset scenarios
scenarios = list_scenarios()
print("Available preset scenarios:")
print("=" * 50)
for i, sid in enumerate(scenarios, 1):
    s = get_scenario(sid)
    print(f"  {i}. {sid}")
    print(f"     {s.name}: {s.description or '(product, region, time, variables)'}")
print()


**Set `SCENARIO_ID`** to one of the IDs listed above to load that scenario. You can then **skip to Step 7 (Estimate size)**. **Set `SCENARIO_ID`** below to `None` to continue with manual selection (Steps 3–6).

In [ ]:
# ============================================
# USER INPUT: Set to None for manual selection, or a scenario ID from the list above
# ============================================
SCENARIO_ID = None  # e.g. "gulf_of_mexico_phy"

if SCENARIO_ID is not None:
    s = get_scenario(SCENARIO_ID)
    SELECTED_PRODUCT_ID = s.product_id
    REGION = s.region
    TIME_RANGE = s.time
    SELECTED_VARIABLES = s.variables
    # Load dataset card and derived info so later steps (estimate, fetch, plot) work
    card = describe(SELECTED_PRODUCT_ID)
    time_start = card.time_coverage_start
    time_end = card.time_coverage_end
    available_variables = [v.canonical_name for v in card.variables]
    variable_info = {v.canonical_name: v for v in card.variables}
    print(f"Scenario loaded: {s.name}")
    print(f"  Product: {SELECTED_PRODUCT_ID}")
    print(f"  Region: lon={REGION['lon']}, lat={REGION['lat']}")
    print(f"  Time: {TIME_RANGE[0]} to {TIME_RANGE[1]}")
    print(f"  Variables: {', '.join(SELECTED_VARIABLES)}")
    print("\nYou can skip to Step 7 (Estimate data size).")
else:
    print("No scenario selected. Continue with Steps 3–6 to choose product, variables, region, and time.")

## Step 3: Select a Dataset and View Its Details

Set `SELECTED_PRODUCT_ID` and inspect the dataset card (coverage, variables, resolution, access notes).


In [ ]:
# ============================================
# USER INPUT: Select your dataset here
# ============================================
# Example:
# SELECTED_PRODUCT_ID = "hycom_glbv0.08_reanalysis_53x"

SELECTED_PRODUCT_ID = "copernicus_phy_reanalysis_001_030"  # Change this to your choice

# Get Dataset Card (metadata)
card = describe(SELECTED_PRODUCT_ID)
print(card)

### Extract Key Info from the Dataset Card

This populates handy variables used later (time coverage, available variables, metadata).


In [ ]:
# Extract time coverage
time_start = card.time_coverage_start
time_end = card.time_coverage_end

# Extract available variables
available_variables = [v.canonical_name for v in card.variables]
variable_info = {v.canonical_name: v for v in card.variables}

print(f"\n📅 Time Coverage: {time_start} to {time_end}")
print(f"⏱️  Temporal Resolution: {card.temporal_resolution}")
print(f"🌍 Spatial Resolution: {card.spatial_resolution}")
print(f"\n📊 Available Variables ({len(available_variables)}):")
for var in available_variables:
    vinfo = variable_info[var]
    dim_type = "2D (surface)" if vinfo.is_2d else "3D (depth-resolved)"
    print(f"  • {var:10s} - {vinfo.description:40s} [{vinfo.units:10s}] ({dim_type})")

## Step 4: Select Variables

Pick one or more variables in `SELECTED_VARIABLES` from the list above.


In [ ]:
# ============================================
# USER INPUT: Select your variables
# ============================================
# Example:
# SELECTED_VARIABLES = ["temp", "sal", "u", "v"]  

SELECTED_VARIABLES = ["temp", "sal", "u", "v", "ssh"]  # Change this to your choice

# Validate selection
invalid_vars = [v for v in SELECTED_VARIABLES if v not in available_variables]
if invalid_vars:
    raise ValueError(f"Invalid variables: {invalid_vars}. Available: {available_variables}")

print(f"✅ Selected {len(SELECTED_VARIABLES)} variable(s): {', '.join(SELECTED_VARIABLES)}")

## Step 5: Define Your Region

You can use a predefined region ID or specify custom lon/lat bounds.

### Step 5a: (Optional) Pick lon/lat visually (interactive globe)

Use the globe to explore coordinates before choosing bounds. Hover points to read lon/lat.


In [ ]:
# Interactive globe to help choose lon/lat bounds
# - Hover points to read lon/lat
# - Optionally overlay a bounding box (if you set BBOX)

# Optional: set a bbox to preview, or leave as None
BBOX = None  # e.g. {"lon": [-80, -70], "lat": [25, 35]}

# Create and show the interactive globe
fig = interactive_globe(bbox=BBOX)
fig.show()


### Step 5b: Available predefined regions

Run the cell below to list region IDs you can use as presets.


In [ ]:
# ============================================
# USER INPUT: Define your region
# ============================================
regions_list = list_regions()
print("Available predefined regions:")
for r in regions_list:
    info = get_region(r)
    print(f"  {r}: {info.name}")
print()

### Step 5c: Choose your region (preset or custom)

Set `USE_PRESET_REGION` **or** edit `CUSTOM_REGION`. Then we'll plot your bounding box on a map.


In [ ]:

# Option A: Use a preset (set to None to use custom bounds below)
USE_PRESET_REGION = "gulf_of_mexico"  # e.g. "monterey_bay"

# Option B: Custom bounds (used when USE_PRESET_REGION is None)
CUSTOM_REGION = {
    "lon": [-80, -70],  # Longitude range [min, max]
    "lat": [25, 35],    # Latitude range [min, max]
}

if USE_PRESET_REGION:
    REGION = get_region_bounds(USE_PRESET_REGION)
    print(f"Using preset region: {USE_PRESET_REGION}")
else:
    REGION = CUSTOM_REGION

print(f"🌍 Selected Region:")
print(f"   Longitude: {REGION['lon'][0]}° to {REGION['lon'][1]}°")
print(f"   Latitude:  {REGION['lat'][0]}° to {REGION['lat'][1]}°")

# Optional: Visualize region on a map
ax = plot_region_bounds(REGION, margin=10)
plt.show()

## Step 6: Select Time Window

Choose `TIME_RANGE` within the dataset coverage printed in Step 3.


In [ ]:
# ============================================
# USER INPUT: Select time window
# ============================================
# Make sure your dates are within: {time_start} to {time_end}

TIME_RANGE = (
    "2014-01-01",  # Start date (YYYY-MM-DD)
    "2014-01-07"   # End date (YYYY-MM-DD)
)

print(f"📅 Selected Time Window:")
print(f"   Start: {TIME_RANGE[0]}")
print(f"   End:   {TIME_RANGE[1]}")
print(f"\nDataset Coverage: {time_start} to {time_end}")

# Validate dates
from datetime import datetime
start_dt = datetime.strptime(TIME_RANGE[0], "%Y-%m-%d")
end_dt = datetime.strptime(TIME_RANGE[1], "%Y-%m-%d")
ds_start_dt = datetime.strptime(time_start, "%Y-%m-%d")
ds_end_dt = datetime.strptime(time_end, "%Y-%m-%d")

if start_dt < ds_start_dt or end_dt > ds_end_dt:
    print(f"\n⚠️  WARNING: Your time range extends outside dataset coverage!")
    print(f"   Adjust your dates to be within {time_start} to {time_end}")
else:
    print(f"\n✅ Time range is valid for this dataset")

## Step 7: Estimate Data Size

Estimate the approximate download size before fetching.


In [ ]:
# Estimate size
size_estimate = provider.estimate_size(
    SELECTED_PRODUCT_ID,
    REGION,
    TIME_RANGE,
    SELECTED_VARIABLES
)

print("📊 Size Estimate:")
print("=" * 60)
if 'estimate_mb' in size_estimate:
    size_mb = size_estimate['estimate_mb']
    size_gb = size_mb / 1024
    print(f"Estimated size: {size_mb:.1f} MB ({size_gb:.2f} GB)")
else:
    print(f"Estimate: {size_estimate}")

if 'note' in size_estimate:
    print(f"\nNote: {size_estimate['note']}")

print(f"\nVariables: {', '.join(SELECTED_VARIABLES)}")
print(f"Region: lon={REGION['lon']}, lat={REGION['lat']}")
print(f"Time: {TIME_RANGE[0]} to {TIME_RANGE[1]}")

## Step 8: Fetch and Cache the Data

**Download and process** your selected data and returns an `xarray.Dataset` (`ds`). The data will be:
1. Fetched from the source (HYCOM/Copernicus)
2. Sanitized (canonical coordinates, units, missing values)
3. Cached locally for fast re-access

⚠️ **Note**: Large datasets may take time to download. The data is cached, so subsequent access will be much faster.

In [ ]:
print("🔄 Fetching data...")
print(f"   Product: {SELECTED_PRODUCT_ID}")
print(f"   Variables: {', '.join(SELECTED_VARIABLES)}")
print(f"   Region: {REGION}")
print(f"   Time: {TIME_RANGE[0]} to {TIME_RANGE[1]}")
print("\nThis may take a while for large datasets...")

# Fetch data
ds = provider.subset(
    product_id=SELECTED_PRODUCT_ID,
    region=REGION,
    time=TIME_RANGE,
    variables=SELECTED_VARIABLES,
    overwrite=False  # Set to True to re-fetch even if cached
)

print("\n✅ Data fetched successfully!")

print(f"Cache location: {CacheStore().root}")


## Optional: Add TEOS‑10 Pressure & Sound Speed (GSW)

If you have fetched **3D physical variables** (`temp`, `sal`) with a `depth` dimension (meters, positive downward) and `lat` coordinate (degrees north), you can optionally add:

- **`pressure`** (sea pressure in dbar), computed from `depth` and `lat`
- **`sound_speed`** (speed of sound in seawater, m/s), computed from:
  - `sal` (Absolute Salinity, g/kg)
  - `temp` (temperature close to Conservative Temperature, °C)
  - `pressure` (dbar)

These fields are computed using the **TEOS‑10 `gsw`** library and are **not stored in the main OceanBench cache** by default; they are derived on the fly in this notebook.

Run the next cell *only if* your `SELECTED_VARIABLES` include `temp` and `sal` and the dataset has a `depth` coordinate (i.e. you requested a 3D product).

In [ ]:
# OPTIONAL: add TEOS‑10 pressure and sound speed using GSW
from oceanbench_data_provider import add_pressure_and_sound_speed

# Requirements:
# - ds has variables "temp" and "sal"
# - ds has coordinate "depth" (in meters, positive downward)
# - ds has coordinate "lat" (in degrees north)
# If these are satisfied, you can safely run this cell.

required = ["temp", "sal"]
missing = [v for v in required if v not in ds.data_vars]
if "depth" not in ds.coords or "lat" not in ds.coords or missing:
    print("⚠️ Cannot add pressure/sound speed automatically.")
    print("   Needed: variables temp & sal, coords depth & lat.")
    print("   Missing:", missing + [c for c in ["depth", "lat"] if c not in ds.coords])
else:
    ds = add_pressure_and_sound_speed(ds)
    print("✅ Added variables: pressure, sound_speed (TEOS‑10, via gsw).")
    print("\nAvailable data variables after adding TEOS‑10 fields:")
    for name in ds.data_vars:
        print(f"  - {name}")

## Step 9: Understand Coordinates (time & depth)

Before plotting, it helps to understand how **time** and **depth** are represented.

### Time
- `ds.time` contains **real datetimes** (not just indices).
- You can select by date: `ds.sel(time='YYYY-MM-DD')`.

### Depth (for 3D variables)
- 2D variables (e.g. `ssh`) have **no** `depth` dimension (surface only).
- 3D variables (e.g. `temp`, `sal`, `u`, `v`) have a `depth` dimension in **meters**.
- When plotting/animating, you typically choose a **depth index** (`0`, `1`, `2`, ...). Index 0 is usually the surface.


### Step 9a: Inspect the dataset structure

Dimensions and coordinates vary by product; this cell prints a quick summary.


In [ ]:
print("📋 Dataset Information:")
print("=" * 60)
print(f"Dimensions: {dict(ds.dims)}")
print(f"\nData Variables: {list(ds.data_vars)}")
print(f"\nCoordinates:")
for coord in ds.coords:
    coord_data = ds.coords[coord]
    if 'time' in coord.lower():
        print(f"  • {coord}: {len(coord_data)} time steps from {coord_data.values[0]} to {coord_data.values[-1]}")
    elif 'lon' in coord.lower():
        print(f"  • {coord}: {len(coord_data)} points from {coord_data.values.min():.2f}° to {coord_data.values.max():.2f}°")
    elif 'lat' in coord.lower():
        print(f"  • {coord}: {len(coord_data)} points from {coord_data.values.min():.2f}° to {coord_data.values.max():.2f}°")
    elif 'depth' in coord.lower():
        print(f"  • {coord}: {len(coord_data)} levels from {coord_data.values.min():.1f}m to {coord_data.values.max():.1f}m")
    else:
        print(f"  • {coord}: {len(coord_data)} values")

### Step 9b: Inspect the time coordinate

This prints time values and indices (useful for choosing `PLOT_TIME_IDX`).


In [ ]:
# Examine the time coordinate
if 'time' in ds.coords:
    time_coord = ds.time
    print(f"Time coordinate: {time_coord.name}")
    print(f"Number of time steps: {len(time_coord)}")
    print(f"\nFirst few time values:")
    for i, t in enumerate(time_coord.values[:5]):
        print(f"  [{i}] {pd.to_datetime(t)}")
    
    print(f"\nLast few time values:")
    for i, t in enumerate(time_coord.values[-5:], len(time_coord) - 5):
        print(f"  [{i}] {pd.to_datetime(t)}")
    
    print(f"\n✅ Time is stored as datetime64 - you can use actual dates for selection!")
else:
    print("⚠️  No time coordinate found in dataset")

### Step 9c: Inspect the depth coordinate (if present)

This prints depth levels with indices (useful for `PLOT_DEPTH_IDX` / `MOVIE_DEPTH_IDX`).


In [ ]:
# Examine the depth coordinate (for 3D variables)
if "depth" in ds.coords:
    depth_coord = ds.depth
    print(f"Depth coordinate: {depth_coord.name}")
    print(f"Number of depth levels: {len(depth_coord)}")
    print(f"\nFirst few depth values (m):")
    for i, d in enumerate(depth_coord.values[:5]):
        print(f"  [{i}] {float(d):.1f} m")
    print(f"\nLast few depth values (m):")
    n_d = len(depth_coord)
    for i, d in enumerate(depth_coord.values[-5:], n_d - 5):
        print(f"  [{i}] {float(d):.1f} m")
    print(f"\n✅ Depth is in meters - use the index in brackets for PLOT_DEPTH_IDX / MOVIE_DEPTH_IDX!")
else:
    print("⚠️  No depth coordinate (dataset has only 2D/surface variables)")

## Step 10: Visualize Data with Proper Time/Depth Labels

Choose a variable `PLOT_VARIABLE` from the dataset `ds` (including any optional TEOS‑10 fields like `pressure` or `sound_speed` if you added them), a time index `PLOT_TIME_IDX`, and optionally a depth index `PLOT_DEPTH_IDX` (for 3D variables), then plot a map.

In [ ]:
# ============================================
# USER INPUT: Select variable, time, and (for 3D vars) depth for plotting
# ============================================
PLOT_VARIABLE = SELECTED_VARIABLES[0]  # Use first selected variable
PLOT_TIME_IDX = 0  # Time index (0 = first time step)
PLOT_DEPTH_IDX = None  # None = surface; for 3D variables use 0, 1, 2, ... for depth level

# Get the actual datetime for this time index
if 'time' in ds.coords:
    actual_time = pd.to_datetime(ds.time.values[PLOT_TIME_IDX])
    print(f"📅 Plotting {PLOT_VARIABLE} at time index {PLOT_TIME_IDX}")
    print(f"   Actual date/time: {actual_time}")
else:
    actual_time = None
    print(f"📅 Plotting {PLOT_VARIABLE} (no time dimension)")

# Print depth level when variable is 3D
if PLOT_VARIABLE in ds.data_vars and "depth" in ds[PLOT_VARIABLE].dims:
    depth_idx = 0 if PLOT_DEPTH_IDX is None else PLOT_DEPTH_IDX
    if "depth" in ds.coords:
        depth_m = float(ds.depth.values[depth_idx])
        print(f"🔽 Plotting {PLOT_VARIABLE} at depth index {depth_idx}")
        print(f"   Actual depth: {depth_m:.1f} m")
    else:
        print(f"🔽 Depth: index {depth_idx}")

In [ ]:
# Plot the map using the library `quicklook_map` helper
ax = quicklook_map(
    ds,
    PLOT_VARIABLE,
    time_idx=PLOT_TIME_IDX,
    depth_idx=PLOT_DEPTH_IDX,
    variable_info=variable_info,
    show_date=True,
)
plt.show()

## Step 11: Create a Movie / Animation

Generate an animated GIF/MP4 `MOVIE_FORMAT` over time for the selected variable `MOVIE_VARIABLE` . For 3D variables, you can set a depth index `MOVIE_DEPTH_IDX` to animate a specific level.


In [ ]:
# ============================================
# USER INPUT: Movie settings
# ============================================
MOVIE_VARIABLE = SELECTED_VARIABLES[0]  # Variable to animate
MOVIE_FORMAT = "gif"  # "gif" or "mp4"
MOVIE_FPS = 2  # Frames per second (lower = slower animation)
MOVIE_DEPTH_IDX = None  # None = surface; for 3D variables use 0, 1, 2, ... for depth level
MOVIE_OUTPUT_PATH = f"{MOVIE_VARIABLE}_animation.{MOVIE_FORMAT}"

print(f"🎬 Creating {MOVIE_FORMAT.upper()} animation...")
print(f"   Variable: {MOVIE_VARIABLE}")
print(f"   Output: {MOVIE_OUTPUT_PATH}")
print(f"   FPS: {MOVIE_FPS}")
print(f"   Time steps: {len(ds.time) if 'time' in ds.coords else 1}")

In [ ]:
# Create the movie using the library `make_movie` helper
if 'time' in ds.coords and len(ds.time) > 1:
    movie_path = make_movie(
        ds,
        MOVIE_VARIABLE,
        MOVIE_OUTPUT_PATH,
        format=MOVIE_FORMAT,
        fps=MOVIE_FPS,
        depth_idx=MOVIE_DEPTH_IDX,
        variable_info=variable_info,
        show_date=True,
    )
    print(f"\n🎉 Animation complete! Check: {movie_path}")
else:
    print("⚠️  Dataset has only one time step - cannot create animation")

## Step 12 (Optional): Inspect the Cache

OceanBench caches requests locally so you can re-run analyses without re-downloading.


In [ ]:
def show_cache_summary():
    """Show a human-readable summary of cached datasets."""
    provider = DataProvider()
    cache_info = provider.cache_status()
    
    print(f"📦 Cache Location: {cache_info['cache_dir']}")
    print(f"📊 Total cached datasets: {len(cache_info['entries'])}\n")
    print("=" * 80)
    
    for i, entry in enumerate(cache_info['entries'], 1):
        req = entry['request']
        product = req.get('product_id', 'unknown')
        vars_list = ', '.join(req.get('variables', []))
        region = req.get('region', {})
        time_range = req.get('time', [])
        
        print(f"\n{i}. {product}")
        print(f"   Variables: {vars_list}")
        print(f"   Region: lon=[{region.get('lon', ['?', '?'])[0]}, {region.get('lon', ['?', '?'])[1]}], "
              f"lat=[{region.get('lat', ['?', '?'])[0]}, {region.get('lat', ['?', '?'])[1]}]")
        print(f"   Time: {time_range[0] if time_range else '?'} to {time_range[-1] if time_range else '?'}")
        print(f"   Cache key: {entry['cache_key']}")

show_cache_summary()